<a href="https://colab.research.google.com/github/M-Abbi/Probability-Statistics-Bootcamp/blob/main/Chi_squared_Distribution_Problem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## The Scenario: Spotting Algorithmic Execution Drift

An Algorithmic Execution desk builds a VWAP (Volume Weighted Average Price) routing model. When a client wants to sell \$50M of Apple (AAPL) stock, the algorithm breaks it into tiny pieces across 4 major exchanges to hide their footprints: **NYSE, NASDAQ, BATS, and IEX**.

Based on structural market liquidity, the quant model assumes a stable distribution of trade fills across these exchanges:
* **NYSE:** 40% ($p_1 = 0.4$)
* **NASDAQ:** 30% ($p_2 = 0.3$)
* **BATS:** 20% ($p_3 = 0.2$)
* **IEX:** 10% ($p_4 = 0.1$)

Over a 2-hour trading window, the desk observes $N = 1000$ executed orders distributed like this:
* **NYSE:** 360 fills ($O_1$)
* **NASDAQ:** 290 fills ($O_2$)
* **BATS:** 230 fills ($O_3$)
* **IEX:** 120 fills ($O_4$)

The desk head needs to determine: *Is this variation just normal statistical noise, or has the market structural dynamic shifted, meaning our execution model is broken?*

---

## The Math: Chi-Squared Goodness-of-Fit Test

To find out, the quant uses a $\chi^2$ Goodness-of-Fit test on the residuals (the differences between observed and expected fills).

### 1. Setting Up the Residuals
First, we calculate the Expected ($E_i$) number of fills based on our model's assumptions for $N = 1000$ trades, where $E_i = N \times p_i$:

* $E_{\text{NYSE}} = 1000 \times 0.40 = 400$
* $E_{\text{NASDAQ}} = 1000 \times 0.30 = 300$
* $E_{\text{BATS}} = 1000 \times 0.20 = 200$
* $E_{\text{IEX}} = 1000 \times 0.10 = 100$

### 2. The Test Statistic Formula
The Chi-Squared test statistic ($\chi^2$) aggregates the squared normalized residuals across all $k$ categories:

$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}$$

Plugging our desk numbers into the formula:

$$\chi^2 = \frac{(360 - 400)^2}{400} + \frac{(290 - 300)^2}{300} + \frac{(230 - 200)^2}{200} + \frac{(120 - 100)^2}{100}$$

$$\chi^2 = \frac{(-40)^2}{400} + \frac{(-10)^2}{300} + \frac{(30)^2}{200} + \frac{(20)^2}{100}$$

$$\chi^2 = 4 + 0.333 + 4.5 + 4 = 12.833$$

### 3. Evaluating the Distribution
Because we have 4 exchanges, our **Degrees of Freedom (df)** is calculated as:

$$\text{df} = k - 1 = 4 - 1 = 3$$

At a standard 95% confidence level ($\alpha = 0.05$), we compare our calculated $\chi^2$ statistic against the critical value from the cumulative chi-squared distribution:

$$\chi^2_{\text{calculated}} = 12.833 > \chi^2_{\text{critical}}(3, 0.05) = 7.815$$

Because our test statistic lands deep in the upper tail ($12.833 > 7.815$), we reject the null hypothesis ($H_0$). The probability ($p\text{-value}$) of seeing this discrepancy by pure chance is:

$$p\text{-value} = P(\chi^2_3 \ge 12.833) \approx 0.005$$

---

## How the IB Desk Uses This

* **Model Risk Management (MRM):** A $p\text{-value}$ of $0.005$ means there is only a 0.5% chance this routing layout happened by pure luck. The desk knows with 99.5% statistical certainty that the market regime has shifted.
* **Algorithmic Recalibration:** The execution platform uses this result to trigger an automatic halt or adaptation loop, dynamically re-weighting the routing engine to avoid over-exposing orders to toxic exchanges.

In [1]:
import numpy as np
import scipy.stats as stats

# 1. Input the observed data and the model's expected theoretical probabilities
observed_fills = np.array([360, 290, 230, 120])
expected_probs = np.array([0.40, 0.30, 0.20, 0.10])

total_trades = observed_fills.sum()
expected_fills = total_trades * expected_probs

# 2. Compute Chi-Squared Statistic and p-value
chi2_stat, p_value = stats.chisquare(f_obs=observed_fills, f_exp=expected_fills)

# 3. Risk Desk Decision Rules
confidence_level = 0.95
significance_threshold = 1 - confidence_level

print("=" * 50)
print("     ALGORITHMIC ROUTING RISK AUDIT             ")
print("=" * 50)
print(f"Calculated Chi-Squared Stat : {chi2_stat:.3f}")
print(f"Model p-value               : {p_value:.5f}")
print(f"Degrees of Freedom          : {len(observed_fills) - 1}")
print("-" * 50)

if p_value < significance_threshold:
    print("STATUS: ALERT! Reject Model Assumptions.")
    print("REASON: Market regime has shifted significantly. Route weights are broken.")
else:
    print("STATUS: PASS. Variance is within normal statistical noise limits.")
print("=" * 50)

     ALGORITHMIC ROUTING RISK AUDIT             
Calculated Chi-Squared Stat : 12.833
Model p-value               : 0.00501
Degrees of Freedom          : 3
--------------------------------------------------
STATUS: ALERT! Reject Model Assumptions.
REASON: Market regime has shifted significantly. Route weights are broken.
